## Basics

In [35]:
!pip install mlxtend

In [36]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [37]:
transactions = [
    ['Milk', 'Bread'],
    ['Milk', 'Diaper', 'Beer'],
    ['Bread', 'Diaper', 'Beer'],
    ['Milk', 'Bread', 'Diaper'],
    ['Milk', 'Bread', 'Beer'],
    ['Milk', 'Eggs'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread', 'Butter'],
    ['Diaper', 'Beer'],
    ['Milk', 'Diaper', 'Eggs'],
    ['Bread', 'Eggs'],
    ['Milk', 'Bread', 'Eggs'],
    ['Milk', 'Beer'],
    ['Bread', 'Diaper'],
    ['Milk', 'Bread', 'Diaper', 'Beer']
]


### Convert data into Apriori format (VERY IMPORTANT)

In [38]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_array, columns=te.columns_)
print(df)


     Beer  Bread  Butter  Diaper   Eggs   Milk
0   False   True   False   False  False   True
1    True  False   False    True  False   True
2    True   True   False    True  False  False
3   False   True   False    True  False   True
4    True   True   False   False  False   True
5   False  False   False   False   True   True
6   False   True    True   False  False  False
7   False   True    True   False  False   True
8    True  False   False    True  False  False
9   False  False   False    True   True   True
10  False   True   False   False   True  False
11  False   True   False   False   True   True
12   True  False   False   False  False   True
13  False   True   False    True  False  False
14   True   True   False    True  False   True


### Step 5: Find frequent itemsets (Apriori)

In [39]:
frequent_items = apriori(
    df,
    min_support=0.4,
    use_colnames=True
)

print(frequent_items)


    support       itemsets
0  0.400000         (Beer)
1  0.666667        (Bread)
2  0.466667       (Diaper)
3  0.666667         (Milk)
4  0.400000  (Milk, Bread)


### Step 6: Generate association rules

In [40]:
rules = association_rules(
    frequent_items,
    metric='confidence',
    min_threshold=0.6
)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])


  antecedents consequents  support  confidence  lift
0      (Milk)     (Bread)      0.4         0.6   0.9
1     (Bread)      (Milk)      0.4         0.6   0.9


### Step 7: Understand output (MOST IMPORTANT)

In [ ]:
{Milk} → {Bread}
confidence = 0.75
#75% of people who bought Milk also bought Bread
#Milk & Bread bought together in 3 out of 5 → support = 0.6


In [41]:
# ============================================================
# APRIORI ALGORITHM – FROM SCRATCH (NO FUNCTIONS)
# Step-by-step, beginner friendly
# ============================================================

# -----------------------------
# STEP 1: Transaction Data
# -----------------------------
transactions = [
    ['Milk', 'Bread', 'Butter'],
    ['Beer', 'Diaper', 'Milk'],
    ['Milk', 'Bread', 'Eggs'],
    ['Beer', 'Diaper'],
    ['Milk', 'Bread', 'Diaper', 'Beer'],
    ['Bread', 'Butter'],
    ['Milk', 'Eggs'],
    ['Bread', 'Eggs'],
    ['Milk', 'Bread', 'Butter', 'Eggs'],
    ['Diaper', 'Beer'],
    ['Milk', 'Bread'],
    ['Milk', 'Beer'],
    ['Bread', 'Butter', 'Eggs'],
    ['Milk', 'Bread', 'Diaper'],
    ['Milk', 'Bread', 'Beer']
]

total_transactions = len(transactions)

# -----------------------------
# STEP 2: Set thresholds
# -----------------------------
min_support = 0.3      # 30%
min_confidence = 0.7  # 70%

# -----------------------------
# STEP 3: Find all unique items
# -----------------------------
items = set()

for transaction in transactions:
    for item in transaction:
        items.add(item)

print("Unique Items:", items)

# -----------------------------
# STEP 4: Generate frequent 1-itemsets (L1)
# -----------------------------
L1 = []

for item in items:
    count = 0
    for transaction in transactions:
        if item in transaction:
            count += 1

    support = count / total_transactions

    if support >= min_support:
        L1.append({item})
        print(f"{item} -> Support: {round(support,2)}")

# -----------------------------
# STEP 5: Generate frequent 2-itemsets (L2)
# -----------------------------
L2 = []

for i in range(len(L1)):
    for j in range(i + 1, len(L1)):
        pair = L1[i] | L1[j]   # union of two sets
        count = 0

        for transaction in transactions:
            if pair.issubset(transaction):
                count += 1

        support = count / total_transactions

        if support >= min_support:
            L2.append(pair)
            print(f"{pair} -> Support: {round(support,2)}")

# -----------------------------
# STEP 6: Generate frequent 3-itemsets (L3)
# -----------------------------
L3 = []

for i in range(len(L2)):
    for j in range(i + 1, len(L2)):
        triple = L2[i] | L2[j]

        if len(triple) == 3:
            count = 0
            for transaction in transactions:
                if triple.issubset(transaction):
                    count += 1

            support = count / total_transactions

            if support >= min_support:
                if triple not in L3:
                    L3.append(triple)
                    print(f"{triple} -> Support: {round(support,2)}")

# -----------------------------
# STEP 7: Combine all frequent itemsets
# -----------------------------
frequent_itemsets = L1 + L2 + L3

print("\nALL FREQUENT ITEMSETS:")
for itemset in frequent_itemsets:
    print(itemset)

# -----------------------------
# STEP 8: Generate Association Rules
# -----------------------------
print("\nASSOCIATION RULES:")

for itemset in frequent_itemsets:
    if len(itemset) >= 2:
        for item in itemset:
            antecedent = {item}
            consequent = itemset - antecedent

            # Calculate support(itemset)
            count_itemset = 0
            for transaction in transactions:
                if itemset.issubset(transaction):
                    count_itemset += 1
            support_itemset = count_itemset / total_transactions

            # Calculate support(antecedent)
            count_antecedent = 0
            for transaction in transactions:
                if antecedent.issubset(transaction):
                    count_antecedent += 1
            support_antecedent = count_antecedent / total_transactions

            confidence = support_itemset / support_antecedent

            if confidence >= min_confidence:
                print(f"{antecedent} -> {consequent} | "
                      f"Support: {round(support_itemset,2)} | "
                      f"Confidence: {round(confidence,2)}")

# -----------------------------
# STEP 9: End
# -----------------------------
print("\nApriori Algorithm Completed Successfully")


Unique Items: {'Eggs', 'Diaper', 'Milk', 'Bread', 'Butter', 'Beer'}
Eggs -> Support: 0.33
Diaper -> Support: 0.33
Milk -> Support: 0.67
Bread -> Support: 0.67
Beer -> Support: 0.4
{'Milk', 'Bread'} -> Support: 0.47

ALL FREQUENT ITEMSETS:
{'Eggs'}
{'Diaper'}
{'Milk'}
{'Bread'}
{'Beer'}
{'Milk', 'Bread'}

ASSOCIATION RULES:
{'Milk'} -> {'Bread'} | Support: 0.47 | Confidence: 0.7
{'Bread'} -> {'Milk'} | Support: 0.47 | Confidence: 0.7

Apriori Algorithm Completed Successfully
